In [1]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_openai import AzureChatOpenAI
from langchain_openai import AzureOpenAIEmbeddings

from langchain_community.vectorstores import FAISS


# --------------------------------------------------
# Load Environment Variables
# --------------------------------------------------
load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = "2024-12-01-preview"
CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_MODEL")
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")




In [ ]:
data_folder = "data/healthcare_policies"

documents = []

for file_name in os.listdir(data_folder):
    if file_name.endswith(".pdf"):
        loader = PyPDFLoader(
            os.path.join(data_folder, file_name)
        )
        documents.extend(loader.load())

print(f"Loaded {len(documents)} pages")



Loaded 5 pages


In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")


# --------------------------------------------------
# Create Embeddings
# --------------------------------------------------
embeddings = AzureOpenAIEmbeddings(
    azure_deployment=EMBEDDING_DEPLOYMENT,
    api_version=AZURE_OPENAI_API_VERSION
)


# --------------------------------------------------
# Build Vector Store
# --------------------------------------------------
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector store created")


# --------------------------------------------------
# LLM
# --------------------------------------------------
llm = AzureChatOpenAI(
    azure_deployment=CHAT_DEPLOYMENT,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0.5
)



Created 20 chunks
Vector store created


In [11]:

# --------------------------------------------------
# RAG Query Function
# --------------------------------------------------
def ask_rag(prompt: str):

    

    response = llm.invoke(prompt)

    print("\n===== Final Answer =====\n")
    print(response.content)




In [12]:
# Semantic Similarity Search
question = "Is Prior Auth needed for Physical therapy? "
print(question)
retrieved_docs = vectorstore.similarity_search(
    question,
    k=2
)
print("\n===== Retrieved Chunks =====\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\nChunk {i}")
    print("-" * 80)
    print(doc.page_content[:500])
context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}

If the answer is not present in the context,
say that the information was not found.
"""
print(prompt)



Is Prior Auth needed for Physical therapy? 

===== Retrieved Chunks =====


Chunk 1
--------------------------------------------------------------------------------
For Gold PPO members, prior authorization is required beginning with physical therapy visit 11. For Silver
HMO members, prior authorization is required beginning with physical therapy visit 7.
SECTION 3: CONTINUED THERAPY REVIEW
Requests for visits beyond the no-authorization threshold should include the diagnosis, baseline functional
status, progress toward measurable goals, treatment frequency and expected duration.
SECTION 4: MEDICAL NECESSITY

Chunk 2
--------------------------------------------------------------------------------
SECTION 2: ADVANCED DIAGNOSTIC IMAGING
Non-emergency MRI, CT and PET procedures require prior authorization. Emergency imaging is exempt
from prior authorization when performed as part of an emergency department encounter.
SECTION 3: PHYSICAL THERAPY
The first 6 physical therapy visits in a be

In [13]:

ask_rag(prompt)


===== Final Answer =====

Yes, prior authorization is needed for physical therapy beginning with the 7th visit.


In [ ]:

# --------------------------------------------------
# Interactive Chat
# --------------------------------------------------




===== Retrieved Chunks =====


Chunk 1
--------------------------------------------------------------------------------
Synthetic training document - no real member data
Page 1
 Silver HMO 2026 - Benefits & Authorization Policy
Document ID
SILVER-HMO-2026
Plan Type
Silver HMO
Effective Date
2026-01-01
Policy Domain
Benefits & Authorization
Status
Active
SECTION 1: POLICY SCOPE
This policy applies to members enrolled in the Silver HMO plan for plan year 2026. Covered services
generally require use of the HMO network except where emergency-care rules apply.
SECTION 2: ADVANCED DIAGNOSTIC IMAGING
Non-emergency MRI, 

Chunk 2
--------------------------------------------------------------------------------
status, progress toward measurable goals, treatment frequency and expected duration.
SECTION 4: MEDICAL NECESSITY
Continued therapy must demonstrate measurable functional improvement or a documented maintenance
need allowed by the member's benefit plan. Services that are solely custodial